# Phase 0 Lab: the OpenSec Analyst on LangGraph

## An agent that answers questions about SEC filings, citing the filing for every claim

OpenSec's analysts spend about 3 hours reading each 10-K or 10-Q by hand. The pilot is an agent that answers their questions in seconds. Before building it, this lab checks that LangGraph gives us what the agent needs: tool calling, a structured answer, memory for follow-up questions and MLflow tracing. We try it on AMD's and NVIDIA's real filings.

The question, method and pass bar are in `labs/BRIEF.md`.

## 5 parts:

PART 1: Fetch the filings from SEC EDGAR

PART 2: XBRL facts, the source of truth for figures

PART 3: Passages and hybrid search

PART 4: The agent

PART 5: Score the samples against the pass bar

> **Before you start**
>
> This lab calls OpenAI for embeddings and for the agent, so it costs a little. It needs `OPENAI_API_KEY` and `SEC_USER_AGENT` in `.env`, and MLflow running (`docker compose up -d`). Run it from `labs/`, top to bottom. The first run downloads the filings and embeds them; later runs read both from the cache in `data/filings/` and only call the model.

In [1]:
import hashlib
import json
import re
import statistics
import time
import uuid
import warnings
from collections import defaultdict, deque
from collections.abc import Iterator
from dataclasses import dataclass
from datetime import date
from decimal import ROUND_HALF_UP, Decimal
from pathlib import Path
from typing import Any, Literal, Self

import httpx
import mlflow
import numpy as np
import pandas as pd
import tiktoken
import yaml
from bs4 import BeautifulSoup, Tag, XMLParsedAsHTMLWarning
from IPython.display import Markdown, display
from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy
from langchain_core.messages import AIMessage, BaseMessage, HumanMessage
from langchain_core.runnables import RunnableConfig
from langchain_core.tools import BaseTool, tool
from langchain_openai import ChatOpenAI
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer
from mlflow.langchain import autolog as autolog_langchain
from openai import OpenAI
from pydantic import BaseModel, Field, SecretStr, model_validator
from pydantic_settings import BaseSettings, SettingsConfigDict
from rank_bm25 import BM25Okapi

2026/11/13 13:46:30 INFO mlflow.agent.hint: Load the `mlflow-trace-instrumentation` skill at /Users/mehathab/Dev/projects/open-sec/.venv/lib/python3.12/site-packages/mlflow/assistant/skills/mlflow-trace-instrumentation/SKILL.md before writing any tracing code; it ships with this MLflow install. Set MLFLOW_DISABLE_AGENT_HINT=1 to silence this.


## Settings

Keys and model names come from `.env`.

In [2]:
REPO_ROOT = Path.cwd().parent  # the notebook runs from labs/
DATA_DIR = REPO_ROOT / "data"
FILINGS_DIR = DATA_DIR / "filings"
FORMS = ("10-K", "10-Q")


class Settings(BaseSettings):
    model_config = SettingsConfigDict(
        env_file=REPO_ROOT / ".env", env_ignore_empty=True, extra="ignore"
    )

    sec_user_agent: str
    openai_api_key: SecretStr
    model_name: str = "gpt-6.1-terra"
    model_reasoning_effort: str = "medium"
    embedding_model: str = "text-embedding-3-large"
    mlflow_tracking_uri: str = "http://localhost:5001"
    price_input_per_mtok: Decimal | None = None
    price_output_per_mtok: Decimal | None = None


settings = Settings()  # pyright: ignore[reportCallIssue]

Every answer is traced in MLflow (R-06), so we turn on LangChain autologging.

In [3]:
mlflow.set_tracking_uri(settings.mlflow_tracking_uri)
mlflow.set_experiment("phase-0-langgraph-lab")
autolog_langchain()

pd.set_option("display.max_colwidth", 120)
warnings.filterwarnings("ignore", category=XMLParsedAsHTMLWarning)  # some filings are XHTML

companies = yaml.safe_load((DATA_DIR / "tickers.yaml").read_text())["companies"]
print(f"Model: {settings.model_name} at {settings.model_reasoning_effort} reasoning")
print(f"Companies: {', '.join(c['ticker'] for c in companies)}")

Model: gpt-6.1-terra at medium reasoning
Companies: AMD, NVDA


## Part 1: Fetch the filings from SEC EDGAR

EDGAR asks for fewer than 10 requests a second (R-12). Every request waits for a slot in a sliding one-second window of at most 9 requests.

In [4]:
MAX_REQUESTS_PER_SECOND = 9


class RateLimiter:
    """Blocks until a request fits in a sliding one-second window."""

    def __init__(self, max_per_second: int = MAX_REQUESTS_PER_SECOND) -> None:
        self._max = max_per_second
        self._sent: deque[float] = deque()

    def wait(self) -> float:
        while True:
            now = time.monotonic()
            while self._sent and now - self._sent[0] >= 1.0:
                self._sent.popleft()
            if len(self._sent) < self._max:
                self._sent.append(now)
                return now
            time.sleep(1.0 - (now - self._sent[0]))

To check the limit held, we count the most requests that fell in any one second.

In [5]:
def max_requests_in_any_second(sent_at: list[float]) -> int:
    times = sorted(sent_at)
    most, start = 0, 0
    for end, moment in enumerate(times):
        while moment - times[start] >= 1.0:
            start += 1
        most = max(most, end - start + 1)
    return most

The client sends the User-Agent from `.env`, keeps each request's time, and saves each download to disk so a rerun doesn't fetch it again.

In [6]:
class EdgarClient:
    def __init__(self, user_agent: str) -> None:
        self._limiter = RateLimiter()
        self._http = httpx.Client(
            headers={"User-Agent": user_agent, "Accept-Encoding": "gzip, deflate"}, timeout=60
        )
        self.sent_at: list[float] = []

    def get(self, url: str) -> str:
        self.sent_at.append(self._limiter.wait())
        response = self._http.get(url)
        response.raise_for_status()
        return response.text

    def get_cached(self, url: str, path: Path) -> str:
        if not path.exists():
            path.parent.mkdir(parents=True, exist_ok=True)
            path.write_text(self.get(url))
        return path.read_text()


edgar = EdgarClient(settings.sec_user_agent)

A `Filing` is one 10-K or 10-Q: where it lives on EDGAR and where we cache it.

In [7]:
@dataclass(frozen=True)
class Filing:
    ticker: str
    cik: int
    form: str
    accession: str
    filing_date: date
    report_date: date
    primary_document: str

    @property
    def url(self) -> str:
        folder = self.accession.replace("-", "")
        return (
            f"https://www.sec.gov/Archives/edgar/data/{self.cik}/{folder}/{self.primary_document}"
        )

    @property
    def path(self) -> Path:
        return FILINGS_DIR / self.ticker / f"{self.accession}.htm"

Each company's submissions file lists its filings. We keep the 10-Ks and 10-Qs.

In [8]:
def filings_from_submissions(ticker: str, cik: int, submissions: dict[str, Any]) -> list[Filing]:
    recent = submissions["filings"]["recent"]
    rows = zip(
        recent["form"],
        recent["accessionNumber"],
        recent["filingDate"],
        recent["reportDate"],
        recent["primaryDocument"],
        strict=True,
    )
    return [
        Filing(ticker, cik, form, accn, date.fromisoformat(filed), date.fromisoformat(end), doc)
        for form, accn, filed, end, doc in rows
        if form in FORMS
    ]

Then the window from plan decision 4: the last 3 completed fiscal years, plus this fiscal year's 10-Qs.

In [9]:
def filings_in_window(filings: list[Filing], fiscal_years: int = 3) -> list[Filing]:
    annual = sorted((f.report_date for f in filings if f.form == "10-K"), reverse=True)
    start = annual[fiscal_years] if len(annual) > fiscal_years else date.min
    return sorted((f for f in filings if f.report_date > start), key=lambda f: f.report_date)

Now we fetch everything: each company's submissions, its XBRL companyfacts, and every filing in the window. The first run takes a few minutes; a rerun reads the cache.

In [10]:
filings: list[Filing] = []
companyfacts: dict[str, dict[str, Any]] = {}

for company in companies:
    cik = company["cik"]
    folder = FILINGS_DIR / company["ticker"]
    submissions_url = f"https://data.sec.gov/submissions/CIK{cik:010d}.json"
    facts_url = f"https://data.sec.gov/api/xbrl/companyfacts/CIK{cik:010d}.json"

    submissions = json.loads(edgar.get_cached(submissions_url, folder / "submissions.json"))
    companyfacts[company["ticker"]] = json.loads(
        edgar.get_cached(facts_url, folder / "companyfacts.json")
    )
    listed = filings_from_submissions(company["ticker"], cik, submissions)
    for filing in filings_in_window(listed):
        edgar.get_cached(filing.url, filing.path)
        filings.append(filing)

print(f"{len(filings)} filings")

29 filings


Did we stay under the limit? A cached rerun sends no requests, so we keep the request times of the run that filled the cache and check those.

In [11]:
REQUEST_LOG = FILINGS_DIR / "request_log.json"
if edgar.sent_at:
    REQUEST_LOG.write_text(json.dumps(edgar.sent_at))
fetching_run = json.loads(REQUEST_LOG.read_text())

print(f"Requests this run: {len(edgar.sent_at)}")
print(f"Requests in the run that filled the cache: {len(fetching_run)}")
print(f"Most in any one second: {max_requests_in_any_second(fetching_run)}")

Requests this run: 0
Requests in the run that filled the cache: 33
Most in any one second: 9


### Explore the filings

It's always good to look at the data before building on it.

In [12]:
pd.DataFrame(
    [(f.ticker, f.form, f.report_date, f.filing_date, f.accession) for f in filings],
    columns=["ticker", "form", "period end", "filed", "accession"],
)

,ticker,form,period end,filed,accession
0,AMD,10-Q,2023-04-01,2023-05-03,0000002488-23-000076
1,AMD,10-Q,2023-07-01,2023-08-02,0000002488-23-000139
2,AMD,10-Q,2023-09-30,2023-11-01,0000002488-23-000195
3,AMD,10-K,2023-12-30,2024-01-31,0000002488-24-000012
4,AMD,10-Q,2024-03-30,2024-05-01,0000002488-24-000056
5,AMD,10-Q,2024-06-29,2024-07-31,0000002488-24-000123
6,AMD,10-Q,2024-09-28,2024-10-30,0000002488-24-000163
7,AMD,10-K,2024-12-28,2025-02-05,0000002488-25-000012
8,AMD,10-Q,2025-03-29,2025-05-07,0000002488-25-000047
9,AMD,10-Q,2025-06-28,2025-08-06,0000002488-25-000108


## Part 2: XBRL facts, the source of truth for figures

Every figure the agent gives must match XBRL (R-08). Companyfacts holds every value each company has tagged, in every filing.

A `Period` is a fiscal period in the company's own calendar: FY2025, Q3 FY2026, or 9M FY2026 for nine months to date.

In [13]:
@dataclass(frozen=True)
class Period:
    fiscal_year: int
    span: str  # FY, Q1 to Q4, 6M or 9M

    def __str__(self) -> str:
        return f"FY{self.fiscal_year}" if self.span == "FY" else f"{self.span} FY{self.fiscal_year}"


print(Period(2025, "FY"), "|", Period(2026, "Q3"), "|", Period(2026, "9M"))

FY2025 | Q3 FY2026 | 9M FY2026


A `Fact` is one tagged value for one period. A balance-sheet value has no start date, because it is as of the end of the period.

In [14]:
@dataclass(frozen=True)
class Fact:
    id: str
    ticker: str
    concept: str
    label: str
    unit: str
    start: date | None
    end: date
    period: Period
    value: Decimal
    earlier_value: Decimal | None  # set when a later filing revised the value
    source: str  # the filing that first reported the current value, such as "AMD 10-K 2025"

    @property
    def period_text(self) -> str:
        return f"end of {self.period}" if self.start is None else str(self.period)

Chips name filings the way analysts do: a 10-K by its fiscal year, a 10-Q by its quarter.

In [15]:
def filing_name(ticker: str, form: str, period: Period) -> str:
    shown = str(period.fiscal_year) if form == "10-K" else str(period)
    return f"{ticker} {form} {shown}"


print(
    filing_name("AMD", "10-K", Period(2025, "FY")),
    "|",
    filing_name("NVDA", "10-Q", Period(2026, "Q3")),
)

AMD 10-K 2025 | NVDA 10-Q Q3 FY2026


### Which period is a value for?

Companyfacts labels each value with the fiscal year and period of the *filing* that reported it, not of the value's own period. NVIDIA's FY2026 10-K also reports FY2025 and FY2024, all tagged FY2026.

So a period end takes its label from the filing that closes it: the latest period end in a filing is that filing's own fiscal period. First, every value reported in a 10-K or 10-Q:

In [16]:
def filing_entries(concepts: dict[str, Any]) -> Iterator[tuple[str, str, dict[str, Any]]]:
    """(concept, unit, entry) for every value reported in a 10-K or 10-Q."""
    for concept, body in concepts.items():
        for unit, entries in body["units"].items():
            for entry in entries:
                if entry["form"] in FORMS and entry.get("fp"):
                    yield concept, unit, entry

Then each filing's latest period end, labelled with that filing's own fiscal period.

In [17]:
def closing_periods(concepts: dict[str, Any]) -> dict[date, Period]:
    """Maps each period end to the fiscal period of the filing that closes it."""
    latest: dict[str, tuple[date, Period]] = {}
    for _, _, entry in filing_entries(concepts):
        end = date.fromisoformat(entry["end"])
        if entry["accn"] not in latest or end > latest[entry["accn"]][0]:
            latest[entry["accn"]] = (end, Period(entry["fy"], entry["fp"]))
    return dict(latest.values())


closing_by_ticker = {
    ticker: closing_periods(body["facts"]["us-gaap"]) for ticker, body in companyfacts.items()
}

Here is every filing in the window with its fiscal label.

In [18]:
pd.DataFrame(
    [
        (f.report_date, filing_name(f.ticker, f.form, closing_by_ticker[f.ticker][f.report_date]))
        for f in filings
    ],
    columns=["period end", "filing"],
)

,period end,filing
0,2023-04-01,AMD 10-Q Q1 FY2023
1,2023-07-01,AMD 10-Q Q2 FY2023
2,2023-09-30,AMD 10-Q Q3 FY2023
3,2023-12-30,AMD 10-K 2023
4,2024-03-30,AMD 10-Q Q1 FY2024
5,2024-06-29,AMD 10-Q Q2 FY2024
6,2024-09-28,AMD 10-Q Q3 FY2024
7,2024-12-28,AMD 10-K 2024
8,2025-03-29,AMD 10-Q Q1 FY2025
9,2025-06-28,AMD 10-Q Q2 FY2025


A value's length in months then says whether it covers the year, a quarter, or 6 or 9 months to date. A 3-month value that ends at a 10-K's year end is Q4. Values of any other length are left out.

In [19]:
DAYS_PER_MONTH = 30.44


def fact_period(start: date | None, end: date, closing: dict[date, Period]) -> Period | None:
    closed = closing.get(end)
    if closed is None or start is None:
        return closed
    months = round((end - start).days / DAYS_PER_MONTH)
    if months == 12 and closed.span == "FY":
        return Period(closed.fiscal_year, "FY")
    if months == 3:
        return Period(closed.fiscal_year, "Q4" if closed.span == "FY" else closed.span)
    if (months, closed.span) in ((6, "Q2"), (9, "Q3")):
        return Period(closed.fiscal_year, f"{months}M")
    return None

### Revisions

Filings sometimes report a different number for an earlier period. So we group every report of the same concept, unit and dates, oldest first.

In [20]:
Versions = dict[tuple[str, str, str | None, str], list[dict[str, Any]]]


def reported_versions(concepts: dict[str, Any]) -> Versions:
    versions: Versions = defaultdict(list)
    for concept, unit, entry in filing_entries(concepts):
        versions[(concept, unit, entry.get("start"), entry["end"])].append(entry)
    for entries in versions.values():
        entries.sort(key=lambda e: (e["filed"], e["accn"]))
    return versions

The latest value is the one we use. If an earlier filing said something else, we keep that too, so the answer can say it was revised (plan decision 8). A value whose sign alone flipped is a tagging change, not a revision.

In [21]:
def revision(entries: list[dict[str, Any]]) -> tuple[Decimal, Decimal | None, dict[str, Any]]:
    """The current value, the earlier value if it was revised, and the first report of it."""
    values = [Decimal(str(e["val"])) for e in entries]
    current = values[-1]
    earlier = [v for v in values if abs(v) != abs(current)]
    return current, earlier[-1] if earlier else None, entries[values.index(current)]

Putting it together: one `Fact` for each concept, unit and period.

In [22]:
def load_facts(ticker: str, companyfacts: dict[str, Any]) -> list[Fact]:
    concepts = companyfacts["facts"]["us-gaap"]
    closing = closing_periods(concepts)
    loaded = []
    for (concept, unit, start_text, end_text), entries in reported_versions(concepts).items():
        start = date.fromisoformat(start_text) if start_text else None
        end = date.fromisoformat(end_text)
        period = fact_period(start, end, closing)
        if period is None:
            continue
        value, earlier_value, first = revision(entries)
        key = f"{ticker}:{concept}:{unit}:{start}:{end}"
        loaded.append(
            Fact(
                id="f_" + hashlib.sha1(key.encode()).hexdigest()[:8],
                ticker=ticker,
                concept=concept,
                label=concepts[concept]["label"] or concept,
                unit=unit,
                start=start,
                end=end,
                period=period,
                value=value,
                earlier_value=earlier_value,
                source=filing_name(ticker, first["form"], Period(first["fy"], first["fp"])),
            )
        )
    return loaded

In [23]:
all_facts = [f for ticker, body in companyfacts.items() for f in load_facts(ticker, body)]
print(f"{len(all_facts):,} facts")

25,880 facts


### Finding facts by words

The agent asks for "revenue", not for an XBRL concept name it would have to guess. We compare words: lowercase, singular, and with camelCase concept names split apart.

In [24]:
def _words(text: str) -> set[str]:
    spaced = re.sub(r"([a-z])([A-Z])", r"\1 \2", text)
    words = (w for w in re.findall(r"[a-z]+", spaced.lower()) if len(w) > 2)
    return {w[:-1] if w.endswith("s") and not w.endswith("ss") else w for w in words}


_words("NetIncomeLoss Revenues")

{'income', 'loss', 'net', 'revenue'}

`find` ranks facts by the words they share with the query, then by how few extra words their label has, then by how often the concept is reported.

In [25]:
class FactStore:
    def __init__(self, facts: list[Fact]) -> None:
        self.by_id = {f.id: f for f in facts}
        self._words = {f.id: _words(f.label) | _words(f.concept) for f in facts}
        self._label_words = {f.id: _words(f.label) for f in facts}
        self._concept_size: dict[tuple[str, str], int] = defaultdict(int)
        for f in facts:
            self._concept_size[(f.ticker, f.concept)] += 1

    def find(self, ticker: str, query: str, period: str | None, limit: int = 12) -> list[Fact]:
        wanted = _words(query)

        def score(f: Fact) -> tuple[int, int, int]:
            shared = len(wanted & self._words[f.id])
            extra = len(self._label_words[f.id] - wanted)
            return shared, -extra, self._concept_size[(f.ticker, f.concept)]

        candidates = [
            f
            for f in self.by_id.values()
            if f.ticker == ticker.upper()
            and (period is None or str(f.period) == period)
            and score(f)[0]
        ]
        return sorted(candidates, key=score, reverse=True)[:limit]


facts = FactStore(all_facts)

### Explore the facts

Here is what the agent will see when it asks for AMD's revenue in FY2025.

In [26]:
pd.DataFrame(
    [
        (f.id, f.label, f.concept, f.value, f.unit, str(f.period), f.source)
        for f in facts.find("AMD", "revenue", "FY2025")[:5]
    ],
    columns=["id", "label", "concept", "value", "unit", "period", "source"],
)

,id,label,concept,value,unit,period,source
0,f_ac8c773b,"Revenue, Remaining Performance Obligation, Amount",RevenueRemainingPerformanceObligation,315000000,USD,FY2025,AMD 10-K 2025
1,f_5b21cdca,"Revenue from Contract with Customer, Excluding Assessed Tax",RevenueFromContractWithCustomerExcludingAssessedTax,34639000000,USD,FY2025,AMD 10-K 2025


And the most recent values a later filing revised. Most are reclassifications.

In [27]:
revised = [f for f in all_facts if f.earlier_value is not None]
latest_revised = sorted(revised, key=lambda f: f.end, reverse=True)[:8]
print(f"{len(revised):,} revised facts")

pd.DataFrame(
    [(f.ticker, f.label, f.period_text, f.earlier_value, f.value) for f in latest_revised],
    columns=["ticker", "label", "period", "first reported", "now"],
)

1,197 revised facts


,ticker,label,period,first reported,now
0,NVDA,"Debt Securities, Available-for-sale, Continuous Unrealized Loss Position, Less than 12 Months",end of FY2026,13132000000,13100000000
1,AMD,"Employee-related Liabilities, Current",end of FY2025,1454000000,1194000000
2,AMD,Increase (Decrease) in Accrued Liabilities and Other Operating Liabilities,9M FY2025,-145000000,-998000000
3,AMD,"Other Comprehensive Income (Loss), Cash Flow Hedge, Gain (Loss), after Reclassification and Tax",9M FY2025,56000000,52000000
4,AMD,"Other Comprehensive Income (Loss), Cash Flow Hedge, Gain (Loss), after Reclassification and Tax",Q3 FY2025,-23000000,-24000000
5,AMD,Other Noncash Income (Expense),9M FY2025,-29000000,-90000000
6,AMD,Other Nonoperating Income (Expense),9M FY2025,219000000,-3000000
7,AMD,Other Nonoperating Income (Expense),Q3 FY2025,82000000,1000000


## Part 3: Passages and hybrid search

Risks and explanations live in the filing text, not in XBRL. Each filing becomes a list of text blocks, one per paragraph or table.

Table cells keep "$", "(" and "%" next to their numbers, so a quote of a figure matches the filing.

In [28]:
OPENING_CELLS = {"$", "(", "($"}
CLOSING_CELLS = {"%", ")", "%)"}


def _clean(text: str) -> str:
    return re.sub(r"\s+", " ", text.replace("\xa0", " ")).strip()


def _row_text(cells: list[str]) -> str:
    joined: list[str] = []
    for cell in cells:
        if joined and (cell in CLOSING_CELLS or joined[-1] in OPENING_CELLS):
            joined[-1] += cell
        else:
            joined.append(cell)
    return " | ".join(joined)


_row_text(["Net revenue", "$", "34,639", "(", "12", "%)"])

'Net revenue | $34,639 | (12%)'

A table becomes one block, a row per line.

In [29]:
def _table_text(table: Tag) -> str:
    rows = []
    for row in table.find_all("tr"):
        cells = [_clean(cell.get_text(" ")) for cell in row.find_all(["td", "th"])]
        if any(cells):
            rows.append(_row_text([c for c in cells if c]))
    return "\n".join(rows)

Everything else becomes one block per innermost paragraph. Hidden elements, which hold XBRL data rather than text, are dropped.

In [30]:
BLOCK_TAGS = ["p", "div", "li", "h1", "h2", "h3", "h4", "h5", "h6", "table"]


def filing_blocks(html: str) -> list[str]:
    soup = BeautifulSoup(html, "lxml")
    for hidden in soup.select('[style*="display:none"], [style*="display: none"]'):
        hidden.decompose()
    blocks = []
    for element in soup.find_all(BLOCK_TAGS):
        if element.find_parent("table"):
            continue
        if element.name == "table":
            text = _table_text(element)
        elif element.find(BLOCK_TAGS):
            continue
        else:
            text = _clean(element.get_text(" "))
        if text:
            blocks.append(text)
    return blocks

Parsing is slow, so each filing's blocks are cached next to it as text.

In [31]:
BLOCK_SEPARATOR = "\n\n"


def cached_blocks(filing: Filing) -> list[str]:
    path = filing.path.with_suffix(".txt")
    if not path.exists():
        path.write_text(BLOCK_SEPARATOR.join(filing_blocks(filing.path.read_text())))
    return path.read_text().split(BLOCK_SEPARATOR)

Here is the start of AMD's latest 10-K, block by block.

In [32]:
latest_amd_10k = max(
    (f for f in filings if f.ticker == "AMD" and f.form == "10-K"), key=lambda f: f.report_date
)
amd_blocks = cached_blocks(latest_amd_10k)
print(f"{len(amd_blocks):,} blocks")

for block in amd_blocks[:8]:
    print("-", block[:120])

1,147 blocks
- UNITED STATES SECURITIES AND EXCHANGE COMMISSION
- Washington, D.C. 20549
- FORM 10-K
- (Mark One)
- ☒ | ANNUAL REPORT PURSUANT TO SECTION 13 OR 15(d) OF THE SECURITIES EXCHANGE ACT OF 1934.
For the fiscal year ended Dece
- Commission File Number 001-07882
- ADVANCED MICRO DEVICES, INC.
- (Exact name of registrant as specified in its charter)


### Sections and chunks

Item headings ("Item 1A. Risk Factors") split the blocks into sections. A 10-Q's Items carry their Part, because Part I and Part II both have an Item 1. A heading is a short block with no line breaks (tables have line breaks).

In [33]:
PART_HEADING = re.compile(r"^PART\s+(IV|III|II|I)\b", re.IGNORECASE)
ITEM_HEADING = re.compile(r"^ITEM\s+(\d{1,2}[A-C]?)\s*[.:\-–—]?", re.IGNORECASE)
MAX_HEADING_LENGTH = 120
MIN_SECTION_LENGTH = 300


def _is_heading(block: str) -> bool:
    return len(block) <= MAX_HEADING_LENGTH and "\n" not in block


def split_sections(blocks: list[str], form: str) -> list[tuple[str, list[str]]]:
    sections: list[tuple[str, list[str]]] = [("Cover", [])]
    part = ""
    for block in blocks:
        if _is_heading(block) and (part_match := PART_HEADING.match(block)):
            part = part_match.group(1).upper()
        elif _is_heading(block) and (item_match := ITEM_HEADING.match(block)):
            item = f"Item {item_match.group(1).upper()}"
            sections.append((f"Part {part}, {item}" if form == "10-Q" else item, []))
        sections[-1][1].append(block)
    return [s for s in sections if len(BLOCK_SEPARATOR.join(s[1])) >= MIN_SECTION_LENGTH]

Each section is cut into chunks of about 2,000 characters. A chunk never splits a block, so a table stays whole.

In [34]:
CHUNK_SIZE = 2000


def chunk_blocks(blocks: list[str]) -> list[str]:
    chunks: list[list[str]] = [[]]
    size = 0
    for block in blocks:
        if chunks[-1] and size + len(block) > CHUNK_SIZE:
            chunks.append([])
            size = 0
        chunks[-1].append(block)
        size += len(block) + len(BLOCK_SEPARATOR)
    return [BLOCK_SEPARATOR.join(chunk) for chunk in chunks]

Here are the sections of AMD's latest 10-K.

In [35]:
pd.DataFrame(
    [
        (item, len(section), len(chunk_blocks(section)))
        for item, section in split_sections(amd_blocks, latest_amd_10k.form)
    ],
    columns=["section", "blocks", "chunks"],
)

,section,blocks,chunks
0,Cover,37,4
1,Item 1,180,36
2,Item 1A,248,77
3,Item 1C,22,3
4,Item 2,3,1
5,Item 5,14,2
6,Item 7,124,22
7,Item 7A,11,4
8,Item 8,410,65
9,Item 9A,15,4


### Passages

Each chunk becomes a `Passage` with its chip, like `AMD 10-K 2025 · Item 7` (plan decision 6). The chip is what the analyst sees next to each claim.

In [36]:
@dataclass(frozen=True)
class Passage:
    id: str
    ticker: str
    form: str
    fiscal_year: int
    item: str
    chip: str
    text: str


def passages_for(filing: Filing, period: Period) -> list[Passage]:
    name = filing_name(filing.ticker, filing.form, period)
    sections = split_sections(cached_blocks(filing), filing.form)
    chunks = [(item, chunk) for item, blocks in sections for chunk in chunk_blocks(blocks)]
    return [
        Passage(
            id="p_" + hashlib.sha1(f"{filing.accession}:{n}".encode()).hexdigest()[:10],
            ticker=filing.ticker,
            form=filing.form,
            fiscal_year=period.fiscal_year,
            item=item,
            chip=f"{name} · {item}",
            text=chunk,
        )
        for n, (item, chunk) in enumerate(chunks)
    ]

In [37]:
passages = [
    passage
    for filing in filings
    for passage in passages_for(filing, closing_by_ticker[filing.ticker][filing.report_date])
]
print(f"{len(passages):,} passages")

3,888 passages


How many passages each Item has, by company:

In [38]:
by_item = pd.DataFrame(
    [(p.ticker, p.form, p.item) for p in passages], columns=["ticker", "form", "item"]
)
by_item.value_counts().unstack("ticker").fillna(0).astype(int).head(40)

ticker                 AMD  NVDA
form item                       
10-Q Cover              24    33
     Part I, Item 1    318   336
     Part I, Item 2    199   220
     Part I, Item 3     12    11
     Part I, Item 4     15    18
     Part II, Item 1A  933   287
     Part II, Item 2    13    22
     Part II, Item 6    23    22
     Part II, Item 5    12     9
     Part II, Item 1     0     9
10-K Cover              12    21
     Item 1            115    94
     Item 1A           239   209
     Item 1C             9     8
     Item 2              3     3
     Item 5              6    10
     Item 7             63    66
     Item 7A            10     7
     Item 8            194     0
     Item 9A            12     6
     Item 9B             3     3
     Item 10             3     6
     Item 11             3     3
     Item 12             3     3
     Item 14             3     3
     Item 15            40   197
     Item 16             3     6
     Item 9C             0     3
     Item 13             0     3

### Embeddings

Search combines meaning and keywords (plan decision 7). First, meaning: OpenAI embeddings. The model reads at most 8,191 tokens, so a few long tables are cut, for embedding only.

In [39]:
EMBED_MAX_TOKENS = 8191


class OpenAIEmbedder:
    def __init__(self, api_key: SecretStr, model: str) -> None:
        self._client = OpenAI(api_key=api_key.get_secret_value())
        self._model = model
        self._encoding = tiktoken.encoding_for_model(model)

    def embed(self, texts: list[str]) -> np.ndarray:
        tokens = [self._encoding.encode(text)[:EMBED_MAX_TOKENS] for text in texts]
        result = self._client.embeddings.create(model=self._model, input=tokens)
        return np.array([item.embedding for item in result.data], dtype=np.float32)


embedder = OpenAIEmbedder(settings.openai_api_key, settings.embedding_model)

Embeddings are cached on disk, one file per text, so a rerun embeds nothing new.

In [40]:
EMBED_BATCH = 128


def cached_embeddings(embedder: OpenAIEmbedder, texts: list[str], cache_dir: Path) -> np.ndarray:
    cache_dir.mkdir(parents=True, exist_ok=True)
    paths = [cache_dir / f"{hashlib.sha1(t.encode()).hexdigest()}.npy" for t in texts]
    missing = [i for i, path in enumerate(paths) if not path.exists()]
    for begin in range(0, len(missing), EMBED_BATCH):
        batch = missing[begin : begin + EMBED_BATCH]
        for i, vector in zip(batch, embedder.embed([texts[i] for i in batch]), strict=True):
            np.save(paths[i], vector)
    return np.stack([np.load(path) for path in paths])

In [41]:
started = time.perf_counter()
cache_dir = FILINGS_DIR / "embeddings" / settings.embedding_model
vectors = cached_embeddings(embedder, [p.text for p in passages], cache_dir)
print(f"Embedded {len(passages):,} passages in {time.perf_counter() - started:.1f} s")

Embedded 3,888 passages in 0.8 s


### Hybrid search

BM25 finds the exact terms an analyst uses, like "export controls". Reciprocal rank fusion merges the two rankings: each passage scores 1 / (60 + its rank) in each, and the scores add up. Phase 1 moves this to pgvector and Postgres full-text.

In [42]:
RRF_K = 60


def _tokens(text: str) -> list[str]:
    return re.findall(r"[a-z0-9]+", text.lower())


def _fusion_scores(scores: np.ndarray) -> np.ndarray:
    ranks = np.argsort(np.argsort(-scores)) + 1
    return 1.0 / (RRF_K + ranks)


def _unit_rows(vectors: np.ndarray) -> np.ndarray:
    return vectors / np.linalg.norm(vectors, axis=-1, keepdims=True)


_fusion_scores(np.array([0.9, 0.2, 0.5]))

array([0.01639344, 0.01587302, 0.01612903])

The index filters by ticker, form and fiscal year first, then ranks only the passages that match.

In [43]:
class PassageIndex:
    def __init__(
        self, passages: list[Passage], vectors: np.ndarray, embedder: OpenAIEmbedder
    ) -> None:
        self.passages = passages
        self.by_id = {p.id: p for p in passages}
        self._vectors = _unit_rows(vectors)
        self._keywords = BM25Okapi([_tokens(p.text) for p in passages])
        self._embedder = embedder

    def _matching(self, ticker: str | None, form: str | None, year: int | None) -> list[int]:
        return [
            i
            for i, p in enumerate(self.passages)
            if (ticker is None or p.ticker == ticker.upper())
            and (form is None or p.form == form.upper())
            and (year is None or p.fiscal_year == year)
        ]

    def search(
        self, query: str, ticker: str | None, form: str | None, year: int | None, limit: int = 6
    ) -> list[Passage]:
        matching = self._matching(ticker, form, year)
        if not matching:
            return []
        query_vector = _unit_rows(self._embedder.embed([query])[0])
        similarity = self._vectors[matching] @ query_vector
        keyword = np.asarray(self._keywords.get_scores(_tokens(query)))[matching]
        fused = _fusion_scores(similarity) + _fusion_scores(keyword)
        return [self.passages[matching[i]] for i in np.argsort(-fused)[:limit]]


index = PassageIndex(passages, vectors, embedder)

Let's try a search. The top hits for export controls in NVIDIA's 10-Ks should be in its risk factors, Item 1A.

In [44]:
hits = index.search("export controls on data center GPUs to China", "NVDA", "10-K", None)
[p.chip for p in hits]

['NVDA 10-K 2026 · Item 1A',
 'NVDA 10-K 2026 · Item 1A',
 'NVDA 10-K 2026 · Item 1',
 'NVDA 10-K 2025 · Item 7',
 'NVDA 10-K 2025 · Item 1A',
 'NVDA 10-K 2026 · Item 1A']

## Part 4: The agent

The model never types a figure. It writes `{fact:ID}` or `{calc:ID}`, and Python fills in the value with its unit and period (R-05, R-07, R-08). Math runs in Python too, never in the model.

### Formatting figures

Dollars in millions with one decimal, and negatives in parentheses (`docs/BRAND.md`).

In [45]:
MILLION = Decimal(1_000_000)


def rounded(value: Decimal, places: str) -> Decimal:
    return value.quantize(Decimal(places), rounding=ROUND_HALF_UP)


def _magnitude(value: Decimal, unit: str) -> str:
    size = abs(value)
    if unit == "USD" and size >= MILLION:
        return f"${rounded(size / MILLION, '0.1'):,}M"
    if unit == "USD":
        return f"${rounded(size, '1'):,}"
    if unit == "USD/shares":
        return f"${rounded(size, '0.01'):,}"
    if unit == "percent":
        return f"{rounded(size, '0.1'):,}%"
    if unit == "pure":
        return f"{rounded(size * 100, '0.1'):,}%"
    if unit == "shares" and size >= MILLION:
        return f"{rounded(size / MILLION, '0.1'):,}M shares"
    return f"{rounded(size, '1'):,} {unit}"


def format_value(value: Decimal, unit: str) -> str:
    return f"({_magnitude(value, unit)})" if value < 0 else _magnitude(value, unit)


print(format_value(Decimal("34639000000"), "USD"), format_value(Decimal("-84200000"), "USD"))

$34,639.0M ($84.2M)


### Math in Python

A `Calculation` keeps the facts it came from, so its answer can cite them and say which periods it covers. Changes carry a sign.

In [46]:
Operation = Literal["change", "percent_change", "ratio", "sum"]


@dataclass(frozen=True)
class Calculation:
    id: str
    operation: Operation
    facts: tuple[Fact, ...]
    value: Decimal
    unit: str

    def formatted(self) -> str:
        if self.operation in ("change", "percent_change") and self.value >= 0:
            return f"+{_magnitude(self.value, self.unit)}"
        return format_value(self.value, self.unit)

`calculate` does the math. Change and percent change go from the first fact to the second; ratio is first / second, as a percent.

In [47]:
def _value_and_unit(operation: Operation, inputs: list[Fact]) -> tuple[Decimal, str]:
    if operation == "sum":
        return sum((f.value for f in inputs), Decimal(0)), inputs[0].unit
    if len(inputs) != 2:
        raise ValueError(f"{operation} takes exactly two facts")
    first, second = inputs
    if operation == "change":
        return second.value - first.value, first.unit
    if operation == "percent_change":
        return (second.value - first.value) / abs(first.value) * 100, "percent"
    return first.value / second.value * 100, "percent"


def calculate(operation: Operation, inputs: list[Fact]) -> Calculation:
    units = {f.unit for f in inputs}
    if operation != "ratio" and len(units) > 1:
        raise ValueError(f"facts have different units: {sorted(units)}")
    value, unit = _value_and_unit(operation, inputs)
    key = f"{operation}:{[f.id for f in inputs]}"
    calc_id = "c_" + hashlib.sha1(key.encode()).hexdigest()[:8]
    return Calculation(calc_id, operation, tuple(inputs), value, unit)

Python also writes the period after each figure, from the facts behind it.

In [48]:
def period_phrase(covered: tuple[Fact, ...]) -> str:
    """ "in FY2025", "at the end of FY2025", "from FY2024 to FY2025" or "(AMD FY2025 vs NVDA
    FY2026)"."""
    first, last = covered[0], covered[-1]
    if first.ticker != last.ticker:
        return f"({first.ticker} {first.period_text} vs {last.ticker} {last.period_text})"
    if first.period_text != last.period_text:
        return f"from {first.period_text} to {last.period_text}"
    return f"at the {first.period_text}" if first.start is None else f"in {first.period_text}"

Let's try one: AMD's revenue growth from FY2024 to FY2025.

In [49]:
def amd_revenue(period: str) -> Fact:
    found = facts.find("AMD", "revenue", period)
    return next(
        f for f in found if f.concept == "RevenueFromContractWithCustomerExcludingAssessedTax"
    )


growth = calculate("percent_change", [amd_revenue("FY2024"), amd_revenue("FY2025")])
print(growth.formatted(), period_phrase(growth.facts))

+34.3% from FY2024 to FY2025


### What an answer can cite

`Sources` holds the facts, the passages, and the calculations the agent has made. `figure` turns a placeholder into its formatted value and the facts behind it.

In [50]:
@dataclass
class Sources:
    facts: FactStore
    index: PassageIndex
    calculations: dict[str, Calculation]

    def figure(self, kind: str, ref: str) -> tuple[str, tuple[Fact, ...]] | None:
        if kind == "fact" and (fact := self.facts.by_id.get(ref)):
            return format_value(fact.value, fact.unit), (fact,)
        if kind == "calc" and (calc := self.calculations.get(ref)):
            return calc.formatted(), calc.facts
        return None


sources = Sources(facts, index, calculations={})
sources.figure("fact", amd_revenue("FY2025").id)

('$34,639.0M',
 (Fact(id='f_5b21cdca', ticker='AMD', concept='RevenueFromContractWithCustomerExcludingAssessedTax', label='Revenue from Contract with Customer, Excluding Assessed Tax', unit='USD', start=datetime.date(2024, 12, 29), end=datetime.date(2025, 12, 27), period=Period(fiscal_year=2025, span='FY'), value=Decimal('34639000000'), earlier_value=None, source='AMD 10-K 2025'),))

### The answer's shape

A claim is one sentence with its citations. A citation is a passage id and a quote copied from it.

In [51]:
class Citation(BaseModel):
    passage_id: str = Field(description="A passage id from search_passages, such as p_1a2b3c4d5e")
    quote: str = Field(description="A short quote copied exactly from that passage")


class Claim(BaseModel):
    text: str = Field(description="One sentence. Write every figure as {fact:ID} or {calc:ID}.")
    citations: list[Citation] = Field(description="Passages that support the claim")


class Table(BaseModel):
    columns: list[str]
    rows: list[list[str]] = Field(description="Cells hold labels or {fact:ID} / {calc:ID}")

The `Answer` is a summary, then a table or bullets, then caveats (R-02).

Its validator runs `check_answer`, defined next. If the answer fails, LangChain's `ToolStrategy` sends the problems back to the model, which answers again in the same run.

In [52]:
class Answer(BaseModel):
    """The OpenSec Analyst's answer: a summary, then a table or bullets, then caveats."""

    kind: Literal["answer", "not_disclosed", "refusal"]
    summary: Claim
    table: Table | None
    bullets: list[Claim]
    caveats: list[str]

    @property
    def claims(self) -> list[Claim]:
        return [self.summary, *self.bullets]

    @property
    def other_texts(self) -> list[str]:
        """Table cells and caveats: text that isn't a claim."""
        cells = [cell for row in self.table.rows for cell in row] if self.table else []
        return [*cells, *self.caveats]

    @property
    def texts(self) -> list[str]:
        return [claim.text for claim in self.claims] + self.other_texts

    @model_validator(mode="after")
    def passes_checks(self) -> Self:
        if problems := check_answer(self, sources):
            raise ValueError("Fix these and answer again:\n- " + "\n- ".join(problems))
        return self

### The checks

Every claim needs a citation or a figure from XBRL. Every passage must exist, and every quote must appear in its passage (R-03).

In [53]:
PLACEHOLDER = re.compile(r"\{(fact|calc):([a-z0-9_]+)\}")


def _squash(text: str) -> str:
    return re.sub(r"\s+", " ", text).strip().lower()


def citation_problems(claim: Claim, sources: Sources) -> list[str]:
    if not claim.citations and not PLACEHOLDER.search(claim.text):
        return [f"No citation for: {claim.text}"]
    problems = []
    for citation in claim.citations:
        passage = sources.index.by_id.get(citation.passage_id)
        if passage is None:
            problems.append(f"Unknown passage {citation.passage_id}")
        elif _squash(citation.quote) not in _squash(passage.text):
            problems.append(f"Quote not in {citation.passage_id}: {citation.quote}")
    return problems

A figure typed as text, rather than a placeholder, is only allowed when it appears in a cited quote.

In [54]:
FIGURE = re.compile(
    r"\$\s?\d[\d,]*(?:\.\d+)?|\b\d[\d,]*(?:\.\d+)?\s?%|\b\d{1,3}(?:,\d{3})+(?:\.\d+)?"
    r"|\b\d+\.\d+|\b\d+(?:\.\d+)?\s?(?:million|billion|thousand)\b"
)


def untraced_figures(texts: list[str], quotes: list[str]) -> list[str]:
    quoted = " ".join(_squash(q) for q in quotes)
    return [
        f"Typed figure {figure} is not from a fact, calculation or quote"
        for text in texts
        for figure in FIGURE.findall(PLACEHOLDER.sub("", text))
        if _squash(figure) not in quoted
    ]


untraced_figures(["Revenue was $34.6 billion, or {fact:f_1}."], quotes=[])

['Typed figure $34.6 is not from a fact, calculation or quote']

And every placeholder must point at a real fact or calculation.

In [55]:
def unknown_ids(texts: list[str], sources: Sources) -> list[str]:
    return [
        f"Unknown id {kind}:{ref}"
        for text in texts
        for kind, ref in PLACEHOLDER.findall(text)
        if sources.figure(kind, ref) is None
    ]

`check_answer` runs them all. Refusals and "not disclosed" answers have nothing to check.

In [56]:
def check_answer(answer: Answer, sources: Sources) -> list[str]:
    """Problems that stop the answer being shown. Empty means it passes."""
    if answer.kind != "answer":
        return []
    problems = []
    if not answer.table and not answer.bullets:
        problems.append("After the summary, add a table or at least one bullet (R-02).")
    for claim in answer.claims:
        problems += citation_problems(claim, sources)
        problems += untraced_figures([claim.text], [c.quote for c in claim.citations])
    all_quotes = [c.quote for claim in answer.claims for c in claim.citations]
    problems += untraced_figures(answer.other_texts, all_quotes)
    problems += unknown_ids(answer.texts, sources)
    return problems

### Rendering the answer

Rendering turns an answer into what the analyst sees. First, the figures: outside a table each one gets its period; in a table the row or column label gives it.

In [57]:
def fill_figures(text: str, sources: Sources, in_table: bool = False) -> str:
    def fill(match: re.Match[str]) -> str:
        figure = sources.figure(*match.groups())
        assert figure is not None, "check_answer rejects unknown ids"
        value, covered = figure
        return value if in_table else f"{value} {period_phrase(covered)}"

    return PLACEHOLDER.sub(fill, text)


fill_figures(
    f"AMD's revenue grew {{calc:{growth.id}}}.", Sources(facts, index, {growth.id: growth})
)

"AMD's revenue grew +34.3% from FY2024 to FY2025."

Then the chips after each claim: one for every passage it quotes and every fact behind its figures.

In [58]:
def cited_facts(texts: list[str], sources: Sources) -> list[Fact]:
    """The facts behind every placeholder, calculations' inputs included, without repeats."""
    found = []
    for text in texts:
        for kind, ref in PLACEHOLDER.findall(text):
            if figure := sources.figure(kind, ref):
                found += figure[1]
    return list(dict.fromkeys(found))


def chips(claim: Claim, sources: Sources) -> str:
    passages = [p for c in claim.citations if (p := sources.index.by_id.get(c.passage_id))]
    names = [p.chip for p in passages]
    names += [f"{f.source} · XBRL" for f in cited_facts([claim.text], sources)]
    return " ".join(f"`{name}`" for name in dict.fromkeys(names))

A revised figure gets a caveat saying what was first reported.

In [59]:
def revision_note(fact: Fact) -> str | None:
    if fact.earlier_value is None:
        return None
    first = format_value(fact.earlier_value, fact.unit)
    now = format_value(fact.value, fact.unit)
    return (
        f"{fact.label} {period_phrase((fact,))} was revised in a later filing: "
        f"first reported as {first}, now {now}."
    )


revision_note(latest_revised[0])

'Debt Securities, Available-for-sale, Continuous Unrealized Loss Position, Less than 12 Months at the end of FY2026 was revised in a later filing: first reported as $13,132.0M, now $13,100.0M.'

A table becomes Markdown rows.

In [60]:
def table_lines(table: Table, sources: Sources) -> list[str]:
    lines = ["| " + " | ".join(table.columns) + " |", "|" + " --- |" * len(table.columns)]
    for row in table.rows:
        lines.append("| " + " | ".join(fill_figures(c, sources, in_table=True) for c in row) + " |")
    return lines

And `render_answer` puts it all together: the summary, then a table or bullets, then caveats.

In [61]:
NOT_DISCLOSED = "Not disclosed in the filings."


def render_answer(answer: Answer, sources: Sources) -> str:
    if answer.kind == "not_disclosed":
        return NOT_DISCLOSED
    if answer.kind == "refusal":
        return answer.summary.text
    lines = [f"{fill_figures(answer.summary.text, sources)} {chips(answer.summary, sources)}", ""]
    if answer.table:
        lines += [*table_lines(answer.table, sources), ""]
    lines += [f"- {fill_figures(b.text, sources)} {chips(b, sources)}" for b in answer.bullets]
    revisions = [note for f in cited_facts(answer.texts, sources) if (note := revision_note(f))]
    if caveats := [*answer.caveats, *revisions]:
        lines += ["", "**Caveats**", *(f"- {c}" for c in caveats)]
    return "\n".join(lines)

### The prompt

The system prompt lists the companies and their fiscal year ends, which come from the filings. NVIDIA's fiscal year ends in January, so its FY2026 is mostly calendar 2025, and the model needs to know that.

In [62]:
def company_lines(filings: list[Filing]) -> str:
    lines = []
    for company in companies:
        ticker = company["ticker"]
        end = max(f.report_date for f in filings if f.ticker == ticker and f.form == "10-K")
        fiscal_year = closing_by_ticker[ticker][end].fiscal_year
        lines.append(
            f"- {company['name']} (ticker {ticker}): fiscal year ends in {end:%B}; "
            f"FY{fiscal_year} ended on {end:%B %-d, %Y}."
        )
    return "\n".join(lines)


print(company_lines(filings))

- Advanced Micro Devices, Inc. (ticker AMD): fiscal year ends in December; FY2025 ended on December 27, 2025.
- NVIDIA Corporation (ticker NVDA): fiscal year ends in January; FY2026 ended on January 25, 2026.


The rules come straight from `AGENTS.md` and `docs/SPEC.md`: cite every claim, never type a figure, do math in Python, say "Not disclosed" rather than guess, and refuse advice.

In [63]:
INSTRUCTIONS = f"""You are the OpenSec Analyst. You answer equity analysts' questions using only \
the 10-K and 10-Q filings of these companies:
{company_lines(filings)}

How to answer:
- Look facts up with get_fact and passages with search_passages. Never answer from memory.
- Write every figure as a placeholder: {{fact:ID}} for an XBRL fact, {{calc:ID}} for a \
calculation. Never type a number yourself; Python fills the figures in.
- Outside a table, Python adds the period after each figure ("in FY2025", "from FY2024 to \
FY2025"), so don't write the period next to a placeholder. In a table, Python shows only the \
value, so label each row or column with its period.
- Do all math with the compute tool, never in your head: changes, percent changes, ratios, sums.
- Every claim needs support: a {{fact:ID}} or {{calc:ID}} in its text, or citations to passages \
with a short quote copied exactly from the passage.
- Reply with a one- or two-sentence summary, then a table (for several figures or a peer \
comparison) or bullets, then caveats (such as differing fiscal calendars or definitions).
- Use each company's own fiscal labels, such as FY2025 or Q3 FY2026.
- If the filings don't contain the answer, set kind to "not_disclosed". Never estimate.
- Refuse investment advice, stock prices, forecasts and anything outside the filings: set kind \
to "refusal", explain politely in the summary, and call no tools.
- For a follow-up such as "and last year?", reuse the company and measure from earlier turns.
"""

### The tools

Each tool returns plain text the model can read. Each is made from `sources`, so it reads the same facts and passages as the checks. `search_passages` finds filing text:

In [64]:
def make_search_passages(sources: Sources) -> BaseTool:
    @tool
    def search_passages(
        query: str,
        ticker: str | None = None,
        form: str | None = None,
        fiscal_year: int | None = None,
    ) -> str:
        """Search filing text, filtered by ticker, form (10-K, 10-Q) and fiscal_year."""
        hits = sources.index.search(query, ticker, form, fiscal_year)
        return "\n\n".join(f"[{p.id}] {p.chip}\n{p.text}" for p in hits) or "No passages found."

    return search_passages

`get_fact` finds XBRL figures:

In [65]:
def make_get_fact(sources: Sources) -> BaseTool:
    @tool
    def get_fact(ticker: str, query: str, period: str | None = None) -> str:
        """Find XBRL facts such as revenue or net income. period is a label like FY2025, Q3 FY2026,
        9M FY2026; it also matches balance-sheet values at the end of that period. Returns
        id | label | value | period."""
        lines = []
        for f in sources.facts.find(ticker, query, period):
            revised = " | revised" if f.earlier_value is not None else ""
            value = format_value(f.value, f.unit)
            lines.append(f"{f.id} | {f.label} ({f.concept}) | {value} | {f.period_text}{revised}")
        return "\n".join(lines) or "No matching facts."

    return get_fact

`compute` does the math and keeps the result, so the answer can cite it as `{calc:ID}`. A bad request goes back to the model as text, so it can fix it.

In [66]:
def make_compute(sources: Sources) -> BaseTool:
    @tool
    def compute(operation: Operation, fact_ids: list[str]) -> str:
        """change or percent_change from the first fact to the second, ratio (first / second, as
        a percent), or sum. Returns the calculation id to cite as {calc:ID}."""
        if unknown := [i for i in fact_ids if i not in sources.facts.by_id]:
            return f"Unknown fact ids: {unknown}"
        try:
            calc = calculate(operation, [sources.facts.by_id[i] for i in fact_ids])
        except ValueError as error:
            return str(error)
        sources.calculations[calc.id] = calc
        return f"{calc.id} | {calc.formatted()} {period_phrase(calc.facts)}"

    return compute

In [67]:
search_passages = make_search_passages(sources)
get_fact = make_get_fact(sources)
compute = make_compute(sources)

Here is what the model sees when it calls `get_fact` for NVIDIA's revenue in Q3 FY2026.

In [68]:
print(get_fact.invoke({"ticker": "NVDA", "query": "revenue", "period": "Q3 FY2026"}))

f_fdc0e598 | Revenues (Revenues) | $57,006.0M | Q3 FY2026
f_b8ec3ab0 | Cost of Revenue (CostOfRevenue) | $15,157.0M | Q3 FY2026
f_ce200b65 | Revenue, Remaining Performance Obligation, Amount (RevenueRemainingPerformanceObligation) | $2,500.0M | end of Q3 FY2026


### LangGraph's create_agent

`create_agent` runs the loop: the model calls tools until it returns an `Answer`. The in-memory checkpointer keeps each conversation, so follow-ups work (Phase 2 swaps in the Postgres checkpointer, decision 9). It only reloads classes it's told about, and this notebook defines `Answer`.

In [69]:
model = ChatOpenAI(
    model=settings.model_name,
    api_key=settings.openai_api_key,
    reasoning_effort=settings.model_reasoning_effort,
)
checkpointer = InMemorySaver(
    serde=JsonPlusSerializer(allowed_msgpack_modules=[("__main__", "Answer")])
)
agent = create_agent(
    model,
    tools=[search_passages, get_fact, compute],
    system_prompt=INSTRUCTIONS,
    response_format=ToolStrategy(Answer),
    checkpointer=checkpointer,
)

First, one question. The model looks up the fact, answers with a placeholder, and Python fills the figure in.

In [70]:
demo: RunnableConfig = {"configurable": {"thread_id": str(uuid.uuid4())}}
state = agent.invoke(
    {"messages": [HumanMessage("What was AMD's net income in fiscal 2025?")]}, demo
)

for message in state["messages"]:
    message.pretty_print()

================================ Human Message =================================

What was AMD's net income in fiscal 2025?
================================== Ai Message ==================================

[{'arguments': '{"ticker":"AMD","query":"net income","period":"FY2025"}', 'call_id': 'call_zBtrtokxwrP8XbB5fYnQAoE9', 'name': 'get_fact', 'type': 'function_call', 'id': 'fc_0c47fa7d76b09476006af75b0c796087d2bf6e8b32352900f2', 'status': 'completed'}]
Tool Calls:
  get_fact (call_zBtrtokxwrP8XbB5fYnQAoE9)
 Call ID: call_zBtrtokxwrP8XbB5fYnQAoE9
  Args:
    ticker: AMD
    query: net income
    period: FY2025
================================= Tool Message =================================
Name: get_fact

f_1fd13d8a | Income Taxes Paid, Net (IncomeTaxesPaidNet) | $884.0M | FY2025
f_ba22b923 | Interest Income (Expense), Net (InterestIncomeExpenseNet) | $215.0M | FY2025
f_79359d19 | Net Income (Loss) Attributable to Parent (NetIncomeLoss) | $4,335.0M | FY2025
f_e20aa1aa | Deferred Income T

[Trace(trace_id='tr-5800e1ba461cf0e2db598a95da24c29b'), Trace(trace_id='tr-2be79f7964e51ee8883fd728b5a7d086')]

And as the analyst sees it:

In [71]:
display(Markdown(render_answer(state["structured_response"], sources)))

AMD reported net income of $4,335.0M in FY2025. `AMD 10-K 2025 · XBRL`

- Net income from continuing operations was $4,269.0M in FY2025, while net income from discontinued operations was $66.0M in FY2025. `AMD 10-K 2025 · XBRL`

### One question, one trace

`ask` runs one question in one MLflow trace, retries included (R-06). It records what the turn used: tools, answer attempts and tokens.

In [72]:
@dataclass
class Result:
    question: str
    answer: Answer
    seconds: float
    tools_used: list[str]
    answer_attempts: int
    input_tokens: int
    output_tokens: int
    trace_id: str | None

A thread holds the whole conversation, so we take only the model's messages since the latest question.

In [73]:
def this_turn(messages: list[BaseMessage]) -> list[AIMessage]:
    asked = max(i for i, m in enumerate(messages) if isinstance(m, HumanMessage))
    return [m for m in messages[asked:] if isinstance(m, AIMessage)]

In [74]:
@mlflow.trace(name="ask")
def ask(question: str, thread_id: str) -> Result:
    config: RunnableConfig = {"configurable": {"thread_id": thread_id}}
    started = time.perf_counter()
    state = agent.invoke({"messages": [HumanMessage(question)]}, config)
    seconds = time.perf_counter() - started

    turn = this_turn(state["messages"])
    calls = [call["name"] for message in turn for call in message.tool_calls]
    usage = [message.usage_metadata for message in turn if message.usage_metadata]
    return Result(
        question=question,
        answer=state["structured_response"],
        seconds=seconds,
        tools_used=[name for name in calls if name != Answer.__name__],
        answer_attempts=calls.count(Answer.__name__),
        input_tokens=sum(u["input_tokens"] for u in usage),
        output_tokens=sum(u["output_tokens"] for u in usage),
        trace_id=mlflow.get_active_trace_id(),
    )

> **Why this matters for OpenSec**
>
> An analyst can only trust an answer they can check. Python, not the model, writes every figure, and every claim carries a chip back to the filing, so a wrong number can't slip in unseen. Each trace in MLflow shows the question, the sources, the steps and the tokens, so a reviewer can see how any answer was made.

## Part 5: Score the samples against the pass bar

The draft samples are in `data/labs/samples.yaml` (plan decision 5): facts, comparisons, risks, a question the filings can't answer, a follow-up and two refusals. Analysts check them and set `reviewed: true`.

In [75]:
samples = yaml.safe_load((DATA_DIR / "labs" / "samples.yaml").read_text())["samples"]
pd.DataFrame(
    [(s["id"], s["kind"], s["question"]) for s in samples], columns=["sample", "kind", "question"]
)

,sample,kind,question
0,S01,fact,What was AMD's revenue in fiscal 2025?
1,S02,fact,What was NVIDIA's net income for fiscal 2026?
2,S03,fact,What was NVIDIA's revenue in the third quarter of fiscal 2026?
3,S04,fact,How much did AMD spend on research and development in fiscal 2025?
4,S05,comparison,By what percentage did AMD's revenue grow from fiscal 2024 to fiscal 2025?
5,S06,comparison,Compare AMD's and NVIDIA's research and development expense in their latest fiscal years.
6,S07,comparison,"Which had higher diluted earnings per share in its latest fiscal year, AMD or NVIDIA?"
7,S08,risk,What does NVIDIA's latest 10-K say about export controls as a risk?
8,S09,risk,What risks does AMD's latest 10-K describe about relying on third-party manufacturers?
9,S10,not_disclosed,How much revenue did NVIDIA earn from Tesla in fiscal 2026?


### Expected figures

Each expected figure is read straight from companyfacts, not through Part 2's period logic, so the score can't share its mistakes. For FY2025 it's the full-year value the FY2025 10-K reports for its own year end. For Q3 FY2026, the three-month value from the Q3 FY2026 10-Q.

In [76]:
@dataclass(frozen=True)
class Expected:
    ticker: str
    concept: str
    period: str
    unit: str
    start: date
    end: date
    value: Decimal

First, the entry the period's own filing reports: its latest end date, and the longest span for a year or the shortest for a quarter.

In [77]:
def own_entry(entries: list[dict[str, Any]], period: str) -> dict[str, Any]:
    span, year = ("FY", period[2:]) if period.startswith("FY") else period.split(" FY")
    own = [
        e for e in entries if e["form"] in FORMS and (e.get("fp"), e.get("fy")) == (span, int(year))
    ]
    end = max(e["end"] for e in own)
    at_end = [e for e in own if e["end"] == end]
    return (min if span == "FY" else max)(at_end, key=lambda e: e["start"])

Then the latest value any filing reported for those dates.

In [78]:
def expected_figure(ticker: str, concept: str, period: str) -> Expected:
    [(unit, entries)] = companyfacts[ticker]["facts"]["us-gaap"][concept]["units"].items()
    chosen = own_entry(entries, period)
    same_dates = [
        e for e in entries if (e.get("start"), e["end"]) == (chosen["start"], chosen["end"])
    ]
    latest = max(same_dates, key=lambda e: e["filed"])
    start, end = date.fromisoformat(chosen["start"]), date.fromisoformat(chosen["end"])
    return Expected(ticker, concept, period, unit, start, end, Decimal(str(latest["val"])))


def expected_figures(sample: dict[str, Any]) -> list[Expected]:
    return [expected_figure(**figure) for figure in sample.get("figures", [])]

Here are the expected figures, for the analysts to check.

In [79]:
pd.DataFrame(
    [
        (sample["id"], e.ticker, e.concept, e.period, e.start, e.end, format_value(e.value, e.unit))
        for sample in samples
        for e in expected_figures(sample)
    ],
    columns=["sample", "ticker", "concept", "period", "start", "end", "expected"],
)

,sample,ticker,concept,period,start,end,expected
0,S01,AMD,RevenueFromContractWithCustomerExcludingAssessedTax,FY2025,2024-12-29,2025-12-27,"$34,639.0M"
1,S02,NVDA,NetIncomeLoss,FY2026,2025-01-27,2026-01-25,"$120,067.0M"
2,S03,NVDA,Revenues,Q3 FY2026,2025-07-28,2025-10-26,"$57,006.0M"
3,S04,AMD,ResearchAndDevelopmentExpense,FY2025,2024-12-29,2025-12-27,"$8,091.0M"
4,S05,AMD,RevenueFromContractWithCustomerExcludingAssessedTax,FY2024,2023-12-31,2024-12-28,"$25,785.0M"
5,S05,AMD,RevenueFromContractWithCustomerExcludingAssessedTax,FY2025,2024-12-29,2025-12-27,"$34,639.0M"
6,S06,AMD,ResearchAndDevelopmentExpense,FY2025,2024-12-29,2025-12-27,"$8,091.0M"
7,S06,NVDA,ResearchAndDevelopmentExpense,FY2026,2025-01-27,2026-01-25,"$18,497.0M"
8,S07,AMD,EarningsPerShareDiluted,FY2025,2024-12-29,2025-12-27,$2.65
9,S07,NVDA,EarningsPerShareDiluted,FY2026,2025-01-27,2026-01-25,$4.90


### The checks for each sample

A figure matches when a cited fact has the same company, dates and value.

In [80]:
def matches(fact: Fact, expected: Expected) -> bool:
    found = (fact.ticker, fact.start, fact.end, fact.value)
    return found == (expected.ticker, expected.start, expected.end, expected.value)


def figures_match(answer: Answer, expected: list[Expected]) -> bool:
    found = cited_facts(answer.texts, sources)
    return all(any(matches(f, e) for f in found) for e in expected)

A calculation matches when the answer cites one with the expected operation, on the expected facts in order.

In [81]:
def calculation_matches(answer: Answer, operation: str, expected: list[Expected]) -> bool:
    for text in answer.texts:
        for kind, ref in PLACEHOLDER.findall(text):
            calc = sources.calculations.get(ref) if kind == "calc" else None
            if (
                calc
                and calc.operation == operation
                and len(calc.facts) == len(expected)
                and all(matches(f, e) for f, e in zip(calc.facts, expected, strict=True))
            ):
                return True
    return False

A risk question must cite a passage from the expected Item, such as NVIDIA's 10-K Item 1A.

In [82]:
def cites_item(answer: Answer, ticker: str, form: str, item: str) -> bool:
    for claim in answer.claims:
        for citation in claim.citations:
            passage = sources.index.by_id.get(citation.passage_id)
            if passage and (passage.ticker, passage.form, passage.item) == (ticker, form, item):
                return True
    return False

`score` picks the checks for the sample's kind. A refusal must call no tools.

In [83]:
def score(sample: dict[str, Any], result: Result) -> dict[str, bool]:
    answer = result.answer
    if sample["kind"] == "refusal":
        return {"refused": answer.kind == "refusal", "no tools": not result.tools_used}
    if sample["kind"] == "not_disclosed":
        return {"not disclosed": answer.kind == "not_disclosed"}

    checks = {
        "structured": answer.kind == "answer",
        "citations pass": not check_answer(answer, sources),
    }
    expected = expected_figures(sample)
    if expected:
        checks["figures match XBRL"] = figures_match(answer, expected)
    if calculation := sample.get("calculation"):
        checks["calculation"] = calculation_matches(answer, calculation["operation"], expected)
    if passage := sample.get("passage"):
        checks["cites expected Item"] = cites_item(answer, **passage)
    return checks

### Running the samples

Cost shows once `PRICE_INPUT_PER_MTOK` and `PRICE_OUTPUT_PER_MTOK` are set in `.env`.

In [84]:
def cost(result: Result) -> Decimal | None:
    if settings.price_input_per_mtok is None or settings.price_output_per_mtok is None:
        return None
    input_cost = result.input_tokens * settings.price_input_per_mtok
    output_cost = result.output_tokens * settings.price_output_per_mtok
    return (input_cost + output_cost) / 1_000_000

Each sample is one MLflow run with its trace, latency, tokens and answer.

In [85]:
def run_sample(sample: dict[str, Any], thread_id: str, name: str) -> dict[str, Any]:
    with mlflow.start_run(run_name=name):
        result = ask(sample["question"], thread_id)
        checks = score(sample, result)
        markdown = render_answer(result.answer, sources)
        mlflow.log_params({"question": sample["question"], "kind": sample["kind"]})
        mlflow.log_params(
            {"model": settings.model_name, "reasoning": settings.model_reasoning_effort}
        )
        mlflow.log_metrics(
            {
                "seconds": result.seconds,
                "input_tokens": result.input_tokens,
                "output_tokens": result.output_tokens,
                "answer_attempts": result.answer_attempts,
                "passed": float(all(checks.values())),
            }
        )
        mlflow.log_text(markdown, "answer.md")
    return {
        "sample": name,
        "kind": sample["kind"],
        "passed": all(checks.values()),
        "failed checks": ", ".join(k for k, ok in checks.items() if not ok),
        "seconds": round(result.seconds, 1),
        "tools": len(result.tools_used),
        "attempts": result.answer_attempts,
        "tokens": result.input_tokens + result.output_tokens,
        "cost": cost(result),
        "trace": result.trace_id,
        "markdown": markdown,
    }

Now every sample. A follow-up runs on the same thread as its question, so the agent has to remember the company and measure.

In [86]:
outcomes = []
for sample in samples:
    thread_id = str(uuid.uuid4())
    outcomes.append(run_sample(sample, thread_id, sample["id"]))
    if follow_up := sample.get("follow_up"):
        follow_up_sample = {**follow_up, "kind": "follow_up"}
        outcomes.append(run_sample(follow_up_sample, thread_id, f"{sample['id']}-follow-up"))

results = pd.DataFrame(outcomes)
results.drop(columns=["markdown"])

🏃 View run S01 at: http://localhost:5001/#/experiments/1/runs/f7669167691445efb930963cf42d34d3
🧪 View experiment at: http://localhost:5001/#/experiments/1


🏃 View run S01-follow-up at: http://localhost:5001/#/experiments/1/runs/37796485f4d34e8d946a3b442f738b70
🧪 View experiment at: http://localhost:5001/#/experiments/1


🏃 View run S02 at: http://localhost:5001/#/experiments/1/runs/e2ce3504527a4497a852de476e2f4e4c
🧪 View experiment at: http://localhost:5001/#/experiments/1


🏃 View run S03 at: http://localhost:5001/#/experiments/1/runs/a18511bc56c94f7bbf842484a9576a56
🧪 View experiment at: http://localhost:5001/#/experiments/1


🏃 View run S04 at: http://localhost:5001/#/experiments/1/runs/ff012fd26c24463f8900638a92eb29b1
🧪 View experiment at: http://localhost:5001/#/experiments/1


🏃 View run S05 at: http://localhost:5001/#/experiments/1/runs/53aae78e4d624e54b3a17a27b4be4156
🧪 View experiment at: http://localhost:5001/#/experiments/1


... (5 outputs left out)

🏃 View run R01 at: http://localhost:5001/#/experiments/1/runs/521299abf88a449183da8e468855ab6a
🧪 View experiment at: http://localhost:5001/#/experiments/1


🏃 View run R02 at: http://localhost:5001/#/experiments/1/runs/b914e981d4eb436cab307f08d5c16ae4
🧪 View experiment at: http://localhost:5001/#/experiments/1


,sample,kind,passed,failed checks,seconds,tools,attempts,tokens,cost,trace
0,S01,fact,True,,10.7,3,2,12475,None,tr-59b69b111acb171aa4472824b385c4f5
1,S01-follow-up,follow_up,True,,4.3,1,1,11343,None,tr-89af9d247c08fccfb3d604fa1d4c1fbe
2,S02,fact,True,,9.1,1,3,6310,None,tr-60466b3517e0f634f71bfa391b9253e2
3,S03,fact,True,,5.7,1,2,3361,None,tr-97af5570694cf73db93ff273bf403851
4,S04,fact,True,,6.2,1,2,4132,None,tr-6f98c75a2de3e3503ca41b26d28d51ab
5,S05,comparison,True,,7.8,4,1,9095,None,tr-68c73b81e8097aa11c2476aeb81d234d
6,S06,comparison,True,,13.8,11,1,35191,None,tr-9581afc7d19b61ad07983f596e710910
7,S07,comparison,True,,8.7,4,1,9411,None,tr-3571fa4499ad11d6f02fdb7eeb4f7760
8,S08,risk,True,,15.6,5,1,15418,None,tr-9e5edc825deab2f51a83dda9b7bf5d47
9,S09,risk,True,,14.2,3,1,8105,None,tr-e107b90b2b75f45b8e69d6e988b231bb


[Trace(trace_id='tr-59b69b111acb171aa4472824b385c4f5'), Trace(trace_id='tr-89af9d247c08fccfb3d604fa1d4c1fbe'), Trace(trace_id='tr-60466b3517e0f634f71bfa391b9253e2'), Trace(trace_id='tr-97af5570694cf73db93ff273bf403851'), Trace(trace_id='tr-6f98c75a2de3e3503ca41b26d28d51ab'), Trace(trace_id='tr-68c73b81e8097aa11c2476aeb81d234d'), Trace(trace_id='tr-9581afc7d19b61ad07983f596e710910'), Trace(trace_id='tr-3571fa4499ad11d6f02fdb7eeb4f7760'), Trace(trace_id='tr-9e5edc825deab2f51a83dda9b7bf5d47'), Trace(trace_id='tr-e107b90b2b75f45b8e69d6e988b231bb')]

### The answers

Each answer as an analyst would see it, with figures filled in by Python and chips for every claim.

In [87]:
for outcome in outcomes:
    display(Markdown(f"### {outcome['sample']}\n\n{outcome['markdown']}"))

### S01

AMD reported net revenue of $34,639.0M in FY2025. `AMD 10-K 2025 · Item 8` `AMD 10-K 2025 · XBRL`

- The figure is reported in AMD's consolidated statement of operations. `AMD 10-K 2025 · Item 8`

### S01-follow-up

AMD reported net revenue of $25,785.0M in FY2024. `AMD 10-K 2025 · Item 8` `AMD 10-K 2024 · XBRL`

- The figure is reported in AMD's consolidated statement of operations. `AMD 10-K 2025 · Item 8`

### S02

NVIDIA's net income was $120,067.0M in FY2026. `NVDA 10-K 2026 · XBRL`

| Fiscal year | Net income |
| --- | --- |
| FY2026 | $120,067.0M |


### S03

NVIDIA's revenue was $57,006.0M in Q3 FY2026. `NVDA 10-Q Q3 FY2026 · XBRL`

| Period | Revenue |
| --- | --- |
| Q3 FY2026 | $57,006.0M |


### S04

AMD reported research and development expense of $8,091.0M in FY2025. `AMD 10-K 2025 · XBRL`

| Expense | FY2025 |
| --- | --- |
| Research and development | $8,091.0M |


### S05

AMD's revenue grew by +34.3% from FY2024 to FY2025. `AMD 10-K 2024 · XBRL` `AMD 10-K 2025 · XBRL`

| Fiscal year | Net revenue |
| --- | --- |
| FY2024 | $25,785.0M |
| FY2025 | $34,639.0M |


... (3 outputs left out)

### S09

AMD says its reliance on outside manufacturers exposes it to supply shortages, production problems and technology delays that could materially hurt its business. `AMD 10-K 2025 · Item 1A`

- Concentration and supply: AMD relies on TSMC for all wafers for its most advanced microprocessor and GPU products, and a shortage could mean lost sales or allocation of limited products among customers. `AMD 10-K 2025 · Item 1A`
- Capacity and pricing: Some suppliers have no long-term commitments beyond existing purchase orders and could prioritize other customers, raise prices, demand prepayments or reduce deliveries. `AMD 10-K 2025 · Item 1A`
- Production and technology: Manufacturing errors or inefficiencies can lower yields and delay deliveries, while suppliers may struggle to implement advanced process or packaging technologies on time or profitably. `AMD 10-K 2025 · Item 1A`
- Disruptions and limited control: AMD cites limited control over schedules, quality and costs, as well as risks from facility damage, power or water shortages, raw-material shortages and intellectual-property misappropriation. `AMD 10-K 2025 · Item 1A`
- Switching suppliers: Moving production can delay launches, reduce yields or product performance, and AMD may not be able to find alternative supply quickly. `AMD 10-K 2025 · Item 1A`
- Geopolitical exposure: AMD warns that changes in relations between China and Taiwan could disrupt its Taiwan-based foundries, manufacturing facilities and subcontractors. `AMD 10-K 2025 · Item 1A`

**Caveats**
- These are risks disclosed in AMD’s annual filing, not predictions that each disruption will occur.

### S10

Not disclosed in the filings.

### R01

I can’t provide NVIDIA’s current stock price; I answer questions using its SEC filings only.

### R02

I can’t advise whether you should buy AMD stock, but I can help you review its filings and financial performance.

## The result

The brief's pass bar, checked in Python. If it passes, LangGraph becomes the agent framework and the result goes in `docs/STATUS.md`.

In [88]:
median_seconds = statistics.median(results["seconds"])
first_time = (results["attempts"] == 1).sum()

print(f"Median latency: {median_seconds:.1f} s; max: {results['seconds'].max():.1f} s")
print(f"Median tokens per answer: {statistics.median(results['tokens']):.0f}")
print(f"Answers that passed the checks first time: {first_time} of {len(results)}")

Median latency: 7.8 s; max: 15.6 s
Median tokens per answer: 8105
Answers that passed the checks first time: 9 of 13


In [89]:
pd.Series(
    {
        "All samples pass their checks": bool(results["passed"].all()),
        "Median latency under 15 s": median_seconds < 15,
        "Every run has an MLflow trace": bool(results["trace"].notna().all()),
    },
    name="passed",
)

All samples pass their checks    True
Median latency under 15 s        True
Every run has an MLflow trace    True
Name: passed, dtype: bool

> **What's next**
>
> Phase 1 moves this into product code: `ingest/` fetches, parses, chunks, embeds and loads the filings into Postgres with pgvector, and reads segment figures from each filing's inline XBRL. Phase 2 builds the agent and the `/ask` route on top.